# MIL-CREDA frente a CREDA — fase uno: la corrida

Este cuaderno corre la campaña y nada más: el pronóstico de costo, la búsqueda del techo de cada familia y la campaña completa, en los dos niveles de contaminación que el informe muestra uno al lado del otro. No arma ninguna tabla ni conclusión — eso vive en `Benchmark_Report_v1.ipynb`, que lee `summary.json`, `runs.jsonl` y el registro de la búsqueda desde `MIL-CREDA/Results/Benchmark/` y nunca vuelve a entrenar nada.

Separar los dos es lo que hace barata una corrección al informe: re-renderizarlo cuesta segundos porque no toca la campaña, en vez del costo de cómputo entero.

In [ ]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


In [ ]:
# The repository was resolved by the cell above -- the forge's own, travelling
# byte for byte -- and this one only uses it. `REPOSITORY` is still the name
# every cell below reads, so nothing below changes.
import os
import sys
from pathlib import Path

REPOSITORY = ROOT
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

In [ ]:
import json
import time

from IPython.display import Markdown, display

from MIL_CREDA_Benchmark import config, harness


def show(text: str) -> None:
    """Una tabla se muestra como tabla, no como texto de ancho fijo.

    Es la misma cadena que va al registro, renderizada. Nada se vuelve a
    calcular acá: si esto y el archivo dijeran cosas distintas, habría dos
    versiones del mismo número y ninguna forma de saber cuál se movió.
    """
    display(Markdown(text))


device = harness.resolve_device()
# La escala configurada decide si esto es un ensayo, y el ensayo decide DÓNDE
# escribe. `config.is_pilot_scale()` es la única lectura de esa regla --- las dos
# constantes que separan una escala de la otra son `EPOCHS` y `SEEDS` ---, y
# escribirla otra vez acá serían dos ortografías de lo mismo.
#
# Sin esto la reducción se construía sin `pilot`, o sea `False`: una corrida de
# tres épocas y una semilla escribía en `Results/Benchmark/`, que es el árbol de
# la corrida completa, y sus números quedaban ahí para que alguien los citara.
ES_ENSAYO = config.is_pilot_scale()
# Y de qué árbol se LEE lo que dejó el paso anterior, que es otra pregunta. Las
# dos son la misma en el recorrido local ---en ensayo todo corre y consume lo del
# ensayo--- y se separan en el ENSAYO REMOTO: ahí la campaña corre a escala
# reducida, en el worker, contra el `ceilings.json` que la búsqueda dejó a escala
# COMPLETA, que es el archivo que la campaña real va a abrir y el único que puede
# tener la forma vieja o el brazo que falta. Fuera de ese modo esto es
# `is_pilot_scale()` y no cambia nada.
ES_ENSAYO_ENTRADA = config.upstream_pilot_scale()
reduction = harness.Reduction(device=str(device), environment=harness.environment(),
                              pilot=ES_ENSAYO)

# Las dos pasadas de la campaña, declaradas donde se declara todo lo demás que
# decide dónde escribe esta corrida.
#
# Una campaña es cada transferencia a UNA tasa --- `results_for` lo dice de sí
# mismo, y el barrido de ruido es la forma opuesta, UNA transferencia a través de
# cada nivel ---, así que correr el nivel contaminado no es una campaña más
# grande ni un experimento nuevo: es una segunda pasada de esta misma forma. La
# campaña no lee nada del barrido y el barrido no gobierna nada de acá; hacer que
# dependiera de él favorecería a una de las dos estrategias, y por eso el barrido
# se mira y no se consulta.
#
# Los dos niveles salen de las dos constantes que el informe ya lee, y no de una
# lista derivada de `CHECKPOINT_LEVELS` ni de `NOISE_LEVELS`: `NOISE_REPORTED`
# declara de sí mismo que es «el nivel contaminado que el informe y el latente
# muestran al lado de 0.0», y esas dos son exactamente las pasadas que existen.
# Derivarlas de `CHECKPOINT_LEVELS` --- que hoy vale la misma pareja --- haría
# que un tercer nivel de pesos, que es una decisión del barrido, le agregara una
# pasada a la campaña sin que nadie la pidiera.
#
# Sin la segunda pasada, las celdas contaminadas del informe y la mitad
# contaminada del latente dicen «no hay corridas»: leen
# `results_for(NOISE_REPORTED, "campaign", ...)`, que ningún paso escribía.
NIVELES = (config.NOISE, config.NOISE_REPORTED)

shape = config.sizing()
print(json.dumps(shape, indent=2))
print()
print(harness.header(reduction))
print()
print("environment:", reduction.environment["platform"],
      "| torch", reduction.environment["torch"],
      "| self-hosted" if reduction.environment["selfHosted"] else "| hosted runtime")
print("escala:", "ensayo" if ES_ENSAYO else "completa",
      "| escribe en", " y ".join(
          str(config.results_for(nivel, reduction.kind, reduction.pilot))
          for nivel in NIVELES))

In [ ]:
# One run, timed, before committing to the whole grid. An estimate of the cost is
# cheaper than the cost, so it happens first.
from MIL_CREDA_Benchmark import bags

material = {role: bags.build(code, config.DATA_CACHE, config.SEEDS[0])
            for role, code in zip(("source", "target"), config.TRANSFERS[0])}
probe = harness.run_one("G", config.TRANSFERS[0], config.SEEDS[0],
                        reduction, device, material)
per_run = probe["seconds"]
# Por las dos pasadas y no por una: la rejilla corre entera una vez por nivel,
# así que lo que esta celda pronostica es `len(NIVELES)` veces la rejilla. Un
# pronóstico de una sola pasada diría la mitad de lo que el cuaderno va a gastar,
# que es peor que no pronosticar nada.
rejilla = shape["runs"] * len(NIVELES)
full = (len(config.ARMS) * len(config.TRANSFERS) * 30 * per_run * 20
        / reduction.epochs) * len(NIVELES)
# The search's own forecast, and it goes first because the search goes first. Its
# epochs are its own — never the pilot's — so the ratio is part of the estimate.
busqueda = shape["search"]
segundos = busqueda["runs"] * per_run * busqueda["epochs"] / reduction.epochs
print(f"one full arm, {reduction.epochs} epochs: {per_run:.1f}s")
print(f"the ceiling search ({busqueda['runs']} runs at {busqueda['epochs']} epochs): "
      f"about {segundos / 60:.0f} min"
      # Anuncia lo que hará la celda que obtiene los techos, dos más abajo, y
      # nada más que eso. En ensayo esa celda LEE el registro del ENSAYO y no
      # busca, así que acá no hay búsqueda que pronosticar. A escala completa sí
      # busca, y entonces la pregunta es por el registro COMPLETO: cada rama
      # pregunta por el archivo que su propia celda va a leer, que es la única
      # forma de que el pronóstico no hable de otra corrida.
      + (("  — en ensayo se lee el registro "
          + ("del ensayo" if ES_ENSAYO_ENTRADA else "de la corrida completa")
          + ", no se busca") if ES_ENSAYO
         else ("" if harness.search_record(pilot=False) is None
               else "  — already on record, skipped")))
print(f"this grid ({shape['runs']} runs x {len(NIVELES)} niveles = {rejilla} "
      f"runs): about {rejilla * per_run / 60:.0f} min")
print(f"at 20 epochs and 30 seeds: about {full / 3600:.0f} h")
del material, probe

## La búsqueda del techo

Antes de comparar nada hay que elegir un escalar: hasta dónde sube la rampa del
término de adaptación. Uno solo para las dos familias iguala el coeficiente y
desiguala el balance —los dos objetivos están separados por un factor `B_src`, así
que un mismo número pone la adaptación en la mayor parte de un objetivo y en una
décima parte del otro— por eso se busca **uno por familia**, y cada derivación
hereda el de la suya.

Es un experimento y se declara como tal. Corre sobre `SEARCH_TRANSFERS` y mide
sobre las bolsas de **validación**, nunca sobre el material del que se lee el
veredicto: elegir por resultado ahí haría que el veredicto informe una decisión que
él mismo ya tomó. La elección se hace por diferencias apareadas dentro de cada
`(semilla, transferencia)`, y un empate va al techo más chico —el mismo resultado
con menos adaptación es la afirmación más débil.

Y corre a **su** escala, no a la del piloto. La rampa sube sobre la fracción de
entrenamiento transcurrida: con tres épocas satura en la segunda y todo techo se
alcanza casi enseguida, así que un techo encontrado ahí describe un paisaje en el
que la campaña no entrena nunca. Es la única parte de este cuaderno que no tiene
escala de piloto, y por eso es la más larga.

Se busca una vez. Si `ceilings.json` ya existe se lee y no se vuelve a buscar: que
el registro exista significa que la búsqueda contestó, y sobrescribir una respuesta
porque alguien quería otra es exactamente el refinanciamiento silencioso que la
campaña se niega a hacer. Para volver a empezar hay que borrarlo a mano.

In [ ]:
from dataclasses import replace

# Los techos vigentes. La `reduction` se reconstruye con lo que dice el registro
# y no con lo que quedó en memoria: `config.CEILINGS` se llena al importar, y si
# la búsqueda corre en este mismo proceso ese mapeo sigue vacío y la campaña se
# negaría con razón.
#
# En ensayo se LEEN y no se buscan. La búsqueda de ensayo es un paso propio
# (`search-pilot`), y su archivo, `ceilings.pilot.json`, es raíz declarada de ese
# paso: una campaña de ensayo que además buscara escribiría en el árbol del paso
# de al lado, que es la falla que la declaración de `produces` existe para hacer
# visible.
#
# Y se leen los de la escala de ENTRADA, nombrada. Esa coordenada decía
# `reduction.pilot` y la razón escrita acá era que «la escala es un modo del
# recorrido entero: en ensayo todo corre y consume lo del ensayo». Sigue siendo
# la regla del recorrido LOCAL y por eso `upstream_pilot_scale()` la devuelve
# tal cual fuera del ensayo remoto --- lo que dejó de ser cierto es el «entero»:
# el ensayo remoto es el paso corriendo a escala reducida en el worker contra lo
# que los pasos anteriores dejaron a escala COMPLETA, y ahí las dos escalas son
# distintas a propósito.
#
# Lo que la razón vieja protegía sigue protegido: en el recorrido local una
# campaña de ensayo NO corre bajo los techos de la búsqueda completa ---medidos
# a veinte épocas, en otro experimento--- mientras `ceilings.pilot.json` queda en
# disco sin que nada lo lea. Lo que cambia es sólo el modo en el que consumir lo
# completo es justamente el punto: un ensayo contra las salidas de otro ensayo
# prueba que el cable lleva corriente y no prueba nada del archivo que la corrida
# real va a abrir.
#
# A escala completa se busca, como siempre: este es el único lugar del recorrido
# donde `ceilings.json` puede nacer, y sacar la búsqueda de acá dejaría a la
# campaña completa sin techos y sin quien se los busque.
#
# Y acá, UNA sola vez, arriba del bucle de niveles: los techos se buscaron en
# limpio y se mantienen fijos en las dos pasadas, así que la contaminada corre
# bajo el coeficiente elegido sin contaminación. Resolverlos adentro del bucle
# daría los mismos números hoy y, a escala completa, volvería a preguntar por el
# registro una vez por nivel: la puerta por la que la búsqueda entera --- nueve
# horas y media que nadie autorizó --- entra sin decir que está entrando. Lo que
# la contaminación le cuesta al techo lo separa `noise-diagnostic`, que re-busca
# a su nivel y no toca este registro.
if ES_ENSAYO:
    reduction = replace(
        reduction,
        ceilings=config.ceilings_on_record(pilot=ES_ENSAYO_ENTRADA),
        ceilingsByTransfer=config.ceilings_by_transfer_on_record(
            pilot=ES_ENSAYO_ENTRADA))
else:
    reduction = harness.with_ceilings_in_force(reduction, device)

## La corrida

Una pasada por nivel: la limpia y la contaminada, la misma rejilla entera las dos veces. Los techos ya están elegidos arriba y no se vuelven a tocar, así que lo único que cambia entre las dos es la tasa que lleva el material de entrenamiento.

Una línea por transferencia, no una por corrida: con treinta semillas la lista
completa serían mil ochocientas líneas y todo lo que dicen ya está en las tablas
de más abajo. Esta salida existe para saber que la campaña sigue viva, nada más.

In [ ]:
seen = {"n": 0}

def progress(line: str) -> None:
    seen["n"] += 1
    if seen["n"] % len(config.ARMS) == 0:
        print(f"  {seen['n']:>5}/{shape['runs']} corridas "
              f"({(time.perf_counter() - started) / 60:.1f} min)")

# Una pasada por nivel, y las dos son LA campaña. `replace` sobre la reducción de
# arriba y no una `Reduction` nueva: los techos ya están adentro de ella,
# resueltos una sola vez antes del bucle, y armar otra acá los volvería a pedir
# --- que a escala completa es la búsqueda entera, lanzada sin autorización.
#
# El contador de progreso se reinicia por pasada porque `shape["runs"]` es el
# tamaño de UNA rejilla: dejarlo correr diría `1200/600`.
resumenes = {}
for nivel in NIVELES:
    pasada = replace(reduction, labelNoise=nivel)
    # De la reducción con la que se está por correr, no de la constante: las tres
    # coordenadas de la primera pasada coinciden con `config.NOISE` por
    # casualidad, y esa casualidad es cómo se lee el árbol equivocado.
    raiz = config.results_for(pasada.labelNoise, pasada.kind, pasada.pilot)
    print(f"\ncampaña a ρ={nivel:g} → {raiz}")
    seen["n"] = 0
    started = time.perf_counter()
    summary = harness.campaign(pasada, device, progress=progress)
    runs = [json.loads(line) for line in
            (raiz / "runs.jsonl").read_text().splitlines() if line.strip()]
    print(f"campaña a ρ={nivel:g} terminada en "
          f"{(time.perf_counter() - started) / 60:.1f} min, {len(runs)} corridas")
    resumenes[f"{nivel:g}"] = summary